# 編集距離とあいまい文字列マッチング (RapidFuzz)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_string/edit_distance.ipynb)

NLP クックブック 03-23 ／ 文字列処理と正規化

表記ゆれやタイプミスを含む文字列同士の近さを測りたい。候補リストから一番近いものを選びたい。

In [1]:
!pip install -q rapidfuzz

In [2]:
from rapidfuzz import fuzz, process
from rapidfuzz.distance import DamerauLevenshtein, JaroWinkler, Levenshtein

for a, b in [("コンピューター", "コンピュータ"), ("自然言語処理", "自然原語処理"), ("recieve", "receive"), ("東京都立大学", "首都大学東京")]:
    print(f"{a} / {b}: 編集距離 {Levenshtein.distance(a, b)}  正規化類似度 {Levenshtein.normalized_similarity(a, b):.2f}"
          f"  転置込み {DamerauLevenshtein.distance(a, b)}  Jaro-Winkler {JaroWinkler.similarity(a, b):.2f}")
print(Levenshtein.editops("recieve", "receive"))

コンピューター / コンピュータ: 編集距離 1  正規化類似度 0.86  転置込み 1  Jaro-Winkler 0.97
自然言語処理 / 自然原語処理: 編集距離 1  正規化類似度 0.83  転置込み 1  Jaro-Winkler 0.91
recieve / receive: 編集距離 2  正規化類似度 0.71  転置込み 1  Jaro-Winkler 0.97
東京都立大学 / 首都大学東京: 編集距離 5  正規化類似度 0.17  転置込み 5  Jaro-Winkler 0.67
Editops([Editop(tag='insert', src_pos=3, dest_pos=3), Editop(tag='delete', src_pos=4, dest_pos=5)], src_len=7, dest_len=7)


In [3]:
# 候補の中から近いものを探す
universities = ["東京大学", "京都大学", "東京工業大学", "東京都立大学", "一橋大学", "大阪大学", "東京科学大学", "東北大学"]
for q in ["東京都率大学", "ひとつばし大学", "東工大"]:
    print(q, process.extract(q, universities, scorer=fuzz.WRatio, limit=2))

東京都率大学 [('東京都立大学', 83.33333333333334, 3), ('東京大学', 80.0, 0)]
ひとつばし大学 [('東京大学', 60.00000000000001, 0), ('京都大学', 60.00000000000001, 1)]
東工大 [('東京工業大学', 66.66666666666667, 2), ('東京大学', 57.14285714285714, 0)]


In [4]:
# 単語単位の編集距離 (リストを渡す)
ref, hyp = "the cat sat on the mat".split(), "the cat sit on mat".split()
print(Levenshtein.distance(ref, hyp), Levenshtein.editops(ref, hyp))

2 Editops([Editop(tag='replace', src_pos=2, dest_pos=2), Editop(tag='delete', src_pos=4, dest_pos=4)], src_len=6, dest_len=5)


- 「東工大」のような略称や「ひとつばし」のような読みの違いは、文字の編集距離では拾えない。読みに変換してから比べるか、別名辞書を使う。
- RapidFuzz は C++ 実装で、数百万件の突き合わせにも使える (`process.cdist` で行列計算)。